# Test 1 (A+B): SMOTE+Tomek Resampling + Class Weights
**Ablation Study** — Diabetes Dataset  
**Strategy A** = SMOTE+Tomek (data resampling)  
**Strategy B** = Class Weights (algorithm weighting)  
**Design choice**: class weights are computed from the *original* (pre-resampling) class distribution and applied on top of the balanced data, creating an intentional over-correction to quantify the additive effect of combining both strategies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    ConfusionMatrixDisplay, classification_report
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

os.makedirs('figures', exist_ok=True)
print("Libraries imported.")

## 1. Dataset Download

In [ ]:
import kagglehub
path = kagglehub.dataset_download("mathchi/diabetes-data-set")
print("Dataset path:", path)

## 2. Load & Inspect

In [ ]:
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f"Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Outcome'].value_counts()
bars = ax.bar(['No Diabetes (0)', 'Diabetes (1)'], counts.values,
              color=['steelblue', 'tomato'], edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+4,
            str(val), ha='center', fontweight='bold')
ax.set_title('Class Distribution — Original Dataset', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('figures/class_distribution_original.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Minority class ratio: {counts[1]/counts.sum():.2%}")

In [ ]:
features = df.columns[:-1].tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(features):
    for outcome, color, label in [(0,'steelblue','No Diabetes'),(1,'tomato','Diabetes')]:
        axes[i].hist(df[df['Outcome']==outcome][col], bins=25, alpha=0.6,
                     color=color, label=label, edgecolor='none')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value'); axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)
fig.suptitle('Feature Distributions by Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, square=True)
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Value Imputation
Zeros in physiological columns (Glucose, BloodPressure, SkinThickness, Insulin, BMI) are biologically impossible — replaced with column medians.

In [ ]:
zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)
imputer = SimpleImputer(strategy='median')
df[zero_cols] = imputer.fit_transform(df[zero_cols])
print("Null values after imputation:", df.isnull().sum().sum())

## 5. Min-Max Normalization

In [ ]:
features = df.columns[:-1].tolist()
X = df[features].values
y = df['Outcome'].values

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
print("Feature range after Min-Max normalization:")
print(pd.DataFrame(X_scaled, columns=features).agg(['min','max']).T)

## 6. Train-Test Split (80 / 20, stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size : {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print(f"Train class counts: {np.bincount(y_train)}")
print(f"Test  class counts: {np.bincount(y_test)}")

## 7. Strategy B — Class Weight Computation
Weights are anchored to the *original* imbalanced distribution so that the penalty on the minority class remains meaningful even after resampling.

In [ ]:
# Compute class weights from ORIGINAL training distribution (pre-resampling)
# Using these on resampled data creates intentional over-correction — the key
# design choice for studying combined A+B effects in the ablation study.
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
class_weight_dict = {0: float(cw[0]), 1: float(cw[1])}
neg = int(np.sum(y_train == 0))
pos = int(np.sum(y_train == 1))
scale_pos_weight = neg / pos   # for XGBoost

print(f"Original train distribution  — Negative: {neg}, Positive: {pos}")
print(f"Class weight dict            : {class_weight_dict}")
print(f"XGBoost scale_pos_weight     : {scale_pos_weight:.4f}")

## 8. Strategy A — SMOTE+Tomek Resampling
SMOTE first oversamples the minority class by interpolation; Tomek Links then removes borderline synthetic samples that lie too close to majority-class instances, cleaning decision boundaries.

In [ ]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

print(f"Before SMOTETomek : {np.bincount(y_train)}")
print(f"After  SMOTETomek : {np.bincount(y_train_res)}")
print(f"New training size : {X_train_res.shape[0]}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, counts, title in zip(axes,
        [np.bincount(y_train), np.bincount(y_train_res)],
        ['Before SMOTETomek', 'After SMOTETomek']):
    ax.bar(['No Diabetes','Diabetes'], counts,
           color=['steelblue','tomato'], edgecolor='white', linewidth=1.2)
    for j, v in enumerate(counts):
        ax.text(j, v+2, str(v), ha='center', fontweight='bold')
    ax.set_title(title, fontsize=12); ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts)+40)
plt.suptitle('Training Class Distribution — SMOTETomek', fontsize=13)
plt.tight_layout()
plt.savefig('figures/smote_tomek_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Model Definitions (with Class Weights)

In [ ]:
# Strategy B — class weights derived from ORIGINAL distribution applied on top of resampling
USE_CLASS_WEIGHTS = True

models = {
    'Decision Tree':       DecisionTreeClassifier(class_weight=class_weight_dict,
                                                   random_state=42),
    'SVM':                 SVC(class_weight=class_weight_dict, probability=True,
                               random_state=42),
    'Gaussian NB':         GaussianNB(),  # handled via sample_weight in fit
    'Logistic Regression': LogisticRegression(class_weight=class_weight_dict,
                                               max_iter=1000, random_state=42),
    'XGBoost':             XGBClassifier(
                               n_estimators=300, max_depth=4,
                               learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8, min_child_weight=1,
                               scale_pos_weight=scale_pos_weight,
                               eval_metric='logloss', random_state=42)
}
print(f"Models defined with class weights: {class_weight_dict}")

## 10. 10-Fold Stratified Cross-Validation (on resampled training data)

In [ ]:
# Manual 10-fold CV — avoids sklearn metadata-routing API changes for sample_weight
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = {}

for name, model_cv in models.items():
    fold_scores = []
    for tr_idx, val_idx in skf.split(X_train_res, y_train_res):
        Xtr, Xval = X_train_res[tr_idx], X_train_res[val_idx]
        ytr, yval = y_train_res[tr_idx], y_train_res[val_idx]
        if name == 'Gaussian NB' and USE_CLASS_WEIGHTS:
            sw = compute_sample_weight(class_weight_dict, ytr)
            model_cv.fit(Xtr, ytr, sample_weight=sw)
        else:
            model_cv.fit(Xtr, ytr)
        fold_scores.append(accuracy_score(yval, model_cv.predict(Xval)))
    cv_results[name] = np.array(fold_scores)
    print(f"{name:<22} CV Accuracy: {cv_results[name].mean():.4f} ± {cv_results[name].std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]
colors = ['#4c72b0','#dd8452','#55a868','#c44e52','#8172b2']
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors,
              edgecolor='white', linewidth=1.2)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0.6, 1.0); ax.set_ylabel('Mean CV Accuracy')
ax.set_title('10-Fold Cross-Validation Accuracy by Model', fontsize=13)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('figures/cv_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Train on Full Resampled Set & Evaluate on Test Set

In [ ]:
results = []
trained = {}

for name, model in models.items():
    # ---- fit ----
    if name == 'Gaussian NB' and USE_CLASS_WEIGHTS:
        sw = compute_sample_weight(class_weight_dict, y_train_res)
        model.fit(X_train_res, y_train_res, sample_weight=sw)
    else:
        model.fit(X_train_res, y_train_res)
    trained[name] = model

    # ---- predict ----
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model'     : name,
        'Accuracy'  : round(accuracy_score(y_test, y_pred),  4),
        'Prec_0'    : round(precision_score(y_test, y_pred, pos_label=0, zero_division=0), 4),
        'Prec_1'    : round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4),
        'Recall_0'  : round(recall_score(y_test, y_pred, pos_label=0, zero_division=0),    4),
        'Recall_1'  : round(recall_score(y_test, y_pred, pos_label=1, zero_division=0),    4),
        'F1_0'      : round(f1_score(y_test, y_pred, pos_label=0, zero_division=0),        4),
        'F1_1'      : round(f1_score(y_test, y_pred, pos_label=1, zero_division=0),        4),
        'Macro_F1'  : round(f1_score(y_test, y_pred, average='macro', zero_division=0),    4),
        'ROC_AUC'   : round(roc_auc_score(y_test, y_prob), 4)
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

## 12. Per-Class Classification Report

In [ ]:
print("=" * 65)
for name, model in trained.items():
    y_pred = model.predict(X_test)
    print(f"\n── {name} ──")
    print(classification_report(y_test, y_pred,
          target_names=['No Diabetes (0)','Diabetes (1)'], zero_division=0))

## 13. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(trained), figsize=(4*len(trained), 4))
if len(trained) == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, trained.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Diab.','Diabetes'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
plt.suptitle('Confusion Matrices — All Models', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#4c72b0','#dd8452','#55a868','#c44e52','#8172b2']
for (name, model), color in zip(trained.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random Classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Metrics Comparison Charts

In [ ]:
metrics_to_plot = ['Accuracy','Macro_F1','ROC_AUC','Prec_1','Recall_1','F1_1']
labels = ['Accuracy','Macro-F1','ROC-AUC','Precision\n(Diabetes)','Recall\n(Diabetes)','F1\n(Diabetes)']
x = np.arange(len(metrics_to_plot))
width = 0.15
colors = ['#4c72b0','#dd8452','#55a868','#c44e52','#8172b2']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, color) in enumerate(zip(results_df.index, colors)):
    vals = [results_df.loc[name, m] for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=name, color=color, edgecolor='white')

ax.set_xticks(x + width*2)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0.3, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison — Test Set', fontsize=13)
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('figures/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model_names = results_df.index.tolist()
x = np.arange(len(model_names))
width = 0.25
colors_class = ['#4c72b0','#dd8452','#55a868']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric_pair, title in zip(
        axes,
        [('Prec_0','Prec_1'),('Recall_0','Recall_1'),('F1_0','F1_1')],
        ['Precision by Class','Recall by Class','F1-Score by Class']):
    m0, m1 = metric_pair
    v0 = [results_df.loc[n, m0] for n in model_names]
    v1 = [results_df.loc[n, m1] for n in model_names]
    ax.bar(x - width/2, v0, width, label='No Diabetes (0)',
           color='steelblue', edgecolor='white')
    ax.bar(x + width/2, v1, width, label='Diabetes (1)',
           color='tomato', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=8)
    ax.set_ylim(0.3, 1.05); ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
plt.suptitle('Per-Class Metrics — All Models', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('figures/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
macro_f1_vals = results_df['Macro_F1'].values
model_names_list = results_df.index.tolist()
colors = ['#4c72b0','#dd8452','#55a868','#c44e52','#8172b2']
bars = ax.bar(model_names_list, macro_f1_vals, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, macro_f1_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0.4, 1.0); ax.set_ylabel('Macro-F1 Score')
ax.set_title('Macro-F1 Score by Model (Primary Ablation Metric)', fontsize=12)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('figures/macro_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. Summary

In [ ]:
best = results_df['Macro_F1'].idxmax()
print(f"Best model by Macro-F1 : {best} ({results_df.loc[best,'Macro_F1']:.4f})")
print(f"Best model by ROC-AUC  : {results_df['ROC_AUC'].idxmax()}")
print()
print(results_df[['Accuracy','Macro_F1','ROC_AUC','Prec_1','Recall_1','F1_1']].to_string())

## Conclusion
Test 1 combines **Strategy A** (SMOTE+Tomek) and **Strategy B** (class weights). The double correction — balancing the data *and* penalising minority-class errors algorithmically — is expected to produce high Recall on the diabetic class. Macro-F1 is the primary metric for comparing across ablation tests because it treats both classes equally, reflecting real-world clinical importance.